In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset 09 — Meningitis Dataset with Missing Values — Kaggle

**Descripción:** Dataset médico para diagnóstico de meningitis bacteriana vs. otras causas. Contiene resultados de análisis de líquido cefalorraquídeo (LCR) y otras variables clínicas. El dataset tiene una cantidad significativa de valores faltantes (de ahí su nombre).

**Tarea:** Clasificación binaria o multiclase — predecir el tipo de meningitis.

**Características:**
- Variables clínicas del paciente y resultados de LCR
- Alta presencia de valores nulos (es una característica del dataset)
- Mezcla de variables numéricas y categóricas

**URL Kaggle:** https://www.kaggle.com/datasets/chantest/meningitis-dataset-with-missing-values

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
print('Librerías cargadas correctamente')

## 1. Carga del Dataset

In [ ]:
ruta = '/content/drive/MyDrive/machine learning/datasets/meningitis.csv'
data = pd.read_csv(ruta)
print(f'Dimensiones: {data.shape[0]} filas x {data.shape[1]} columnas')
print(f'm = {data.shape[0]} registros')
print(f'n = {data.shape[1]} columnas')
data.head()

## 2. Revisión de Tipos de Datos

In [ ]:
print('=== TIPOS DE DATOS ===')
print(data.dtypes)
print(f'\nNuméricas:   {len(data.select_dtypes(include=["number"]).columns)}')
print(f'Categóricas: {len(data.select_dtypes(include=["object"]).columns)}')

## 3. Análisis de Valores Nulos

Este dataset se caracteriza por tener muchos valores faltantes. Se analiza el porcentaje de nulos por columna.

In [ ]:
nulos = data.isnull().sum()
pct_nulos = (nulos / len(data) * 100).round(2)
reporte_nulos = pd.DataFrame({'Nulos':nulos,'Porcentaje(%)':pct_nulos}).sort_values('Porcentaje(%)',ascending=False)
print(f'Total valores nulos: {data.isnull().sum().sum()}')
print(f'Columnas con nulos:  {int((nulos>0).sum())}')
print()
print(reporte_nulos[reporte_nulos['Nulos']>0])

## 4. Estrategia de Imputación

Dado el alto porcentaje de nulos, se aplica:
- Columnas con >80% de nulos: se eliminan
- Columnas numéricas restantes: imputar con mediana
- Columnas categóricas restantes: imputar con moda

In [ ]:
df = data.copy()

# Eliminar columnas con más del 80% de nulos
umbral = 0.80
cols_eliminar = [col for col in df.columns if df[col].isnull().mean() > umbral]
print(f'Columnas eliminadas (>80% nulos): {cols_eliminar}')
df = df.drop(columns=cols_eliminar)

# Imputar restantes
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == object:
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print(f'\nNulos restantes: {df.isnull().sum().sum()}')
print(f'Forma del dataset limpio: {df.shape}')

## 5. Preparación de Variables

In [ ]:
# Identificar variable objetivo (última columna o columna de diagnóstico)
print('Columnas disponibles:')
print(list(df.columns))
print()

# Codificar categóricas
cols_cat = df.select_dtypes(include=['object']).columns
for col in cols_cat:
    df[col] = pd.Categorical(df[col]).codes

# Variable objetivo (ajustar según el dataset)
target_col = df.columns[-1]
print(f'Variable objetivo: {target_col}')
print(df[target_col].value_counts())

y = df[target_col].to_numpy()
X = df[[col for col in df.columns if col != target_col]].to_numpy(dtype=np.float64)
print(f'\nX: {X.shape} | y: {y.shape}')

## 6. Estadísticas Descriptivas

In [ ]:
df.describe()

## 7. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Porcentaje de nulos por columna (original)
pct_top = pct_nulos.sort_values(ascending=False).head(15)
axes[0].barh(pct_top.index, pct_top.values, color='tomato', edgecolor='black')
axes[0].set_title('% Nulos por Columna (Top 15)')
axes[0].set_xlabel('Porcentaje (%)')

# Distribución de la variable objetivo
clases, conteos = np.unique(y, return_counts=True)
axes[1].bar([str(c) for c in clases], conteos, color='steelblue', edgecolor='black')
axes[1].set_title('Distribución de la Variable Objetivo')
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('Número de registros')

plt.tight_layout()
plt.show()

## 8. División 80/20

In [ ]:
np.random.seed(42)
idx = np.random.permutation(len(X))
corte = int(len(X)*0.8)
X_train,X_test = X[idx[:corte]],X[idx[corte:]]
y_train,y_test = y[idx[:corte]],y[idx[corte:]]
print(f'Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}')

## 9. Conclusiones

**Meningitis Dataset** se distingue por su alta presencia de valores faltantes, lo que representa un desafío real de preprocesamiento. Se aplicó una estrategia de eliminación de columnas con más del 80% de nulos e imputación con mediana/moda para el resto. El dataset es adecuado para clasificación binaria o multiclase para predecir el tipo de meningitis. La gestión de valores nulos es la parte más importante del preprocesamiento de este dataset.